# 03 — Validation Exploration

Explores the validation pipeline — perplexity distributions, filter rejection rates, before/after quality comparisons, and the impact of each filter on the dataset.

Run after `python validation/scripts/validate.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
import re
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

DATA_DIR = Path('../data')
CURATED_DIR = DATA_DIR / 'curated'
VALIDATED_DIR = DATA_DIR / 'validated'

## 1. Validation stats overview

In [ ]:
stats_path = VALIDATED_DIR / 'validation_stats.json'

if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)

    total = stats['total']
    kept = stats['kept']
    rejected = total - kept

    print(f"Total documents:          {total:>10,}")
    print(f"Kept:                     {kept:>10,}  ({100*kept/max(total,1):.1f}%)")
    print(f"Rejected:                 {rejected:>10,}  ({100*rejected/max(total,1):.1f}%)")
    print()
    print("Rejection breakdown:")
    for key, val in stats.items():
        if key.startswith('rejected_'):
            reason = key.replace('rejected_', '').replace('_', ' ')
            print(f"  {reason:<30} {val:>8,}  ({100*val/max(total,1):.1f}%)")
else:
    print("validation_stats.json not found — run validate.py first")

## 2. Rejection breakdown chart

In [ ]:
if stats_path.exists():
    rejection_keys = {k: v for k, v in stats.items() if k.startswith('rejected_')}
    labels = [k.replace('rejected_', '').replace('_', ' ') for k in rejection_keys]
    values = list(rejection_keys.values())
    values.append(stats['kept'])
    labels.append('kept')

    colors = ['#C44E52'] * (len(values) - 1) + ['#55A868']
    alphas = [0.7] * (len(values) - 1) + [0.9]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(labels, values, color=colors)
    for bar, alpha in zip(bars, alphas):
        bar.set_alpha(alpha)

    ax.set_xlabel('Documents')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.set_title('Validation Filter Rejection Breakdown', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Load samples — before and after validation

In [ ]:
SAMPLE_SIZE = 5_000

def load_sample(path, n=SAMPLE_SIZE):
    records = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            records.append(json.loads(line))
    return records

curated = load_sample(CURATED_DIR / 'train.jsonl')
validated_path = VALIDATED_DIR / 'train.jsonl'

if validated_path.exists():
    validated = load_sample(validated_path)
    print(f"Curated sample:   {len(curated):,} records")
    print(f"Validated sample: {len(validated):,} records")
else:
    print("Validated data not found — run validate.py first")
    validated = []

## 4. Perplexity distribution

In [ ]:
kenlm_path = DATA_DIR / 'models' / 'en.arpa.bin'

if kenlm_path.exists():
    import kenlm
    model = kenlm.Model(str(kenlm_path))

    def get_perplexity(text):
        try:
            return model.perplexity(text[:2000])
        except:
            return None

    # Sample perplexities by source
    sources = ['wikipedia', 'code', 'common_crawl']
    perplexities = {s: [] for s in sources}

    for record in curated[:2000]:
        src = record.get('source', 'unknown')
        if src in perplexities:
            ppl = get_perplexity(record['text'])
            if ppl and ppl < 10000:  # cap outliers for visualization
                perplexities[src].append(ppl)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    colors = {'wikipedia': '#4C72B0', 'code': '#55A868', 'common_crawl': '#DD8452'}

    for ax, source in zip(axes, sources):
        ppls = perplexities[source]
        if not ppls:
            continue
        threshold = np.percentile(ppls, 90)
        ax.hist(ppls, bins=50, color=colors[source], alpha=0.8, edgecolor='white')
        ax.axvline(threshold, color='red', linestyle='--', label=f'p90={threshold:.0f}')
        ax.set_title(f'{source}\nn={len(ppls):,}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Perplexity')
        ax.set_ylabel('Count')
        ax.legend(fontsize=9)

    fig.suptitle('Perplexity Distribution by Source (p90 = filter threshold)', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Summary
    rows = []
    for source in sources:
        ppls = perplexities[source]
        if ppls:
            rows.append({
                'Source': source,
                'n': len(ppls),
                'p10': int(np.percentile(ppls, 10)),
                'p50': int(np.percentile(ppls, 50)),
                'p90': int(np.percentile(ppls, 90)),
                'p99': int(np.percentile(ppls, 99)),
            })
    print(pd.DataFrame(rows).to_string(index=False))

else:
    print("KenLM model not found at data/models/en.arpa.bin")
    print("Download: wget https://dl.fbaipublicfiles.com/cc_net/lm/en.arpa.bin -O data/models/en.arpa.bin")

## 5. Text length before vs after

In [ ]:
if validated:
    before_lens = [len(r['text']) for r in curated]
    after_lens = [len(r['text']) for r in validated]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(before_lens, bins=60, alpha=0.6, color='#C44E52', label=f'Before validation (n={len(before_lens):,})')
    ax.hist(after_lens, bins=60, alpha=0.6, color='#55A868', label=f'After validation (n={len(after_lens):,})')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.set_title('Text Length — Before vs After Validation', fontsize=12, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Before — mean: {np.mean(before_lens):.0f} chars, median: {np.median(before_lens):.0f} chars")
    print(f"After  — mean: {np.mean(after_lens):.0f} chars, median: {np.median(after_lens):.0f} chars")

## 6. Sample high vs low perplexity documents

In [ ]:
if kenlm_path.exists():
    # Score a sample of common crawl documents
    cc_records = [r for r in curated if r.get('source') == 'common_crawl'][:500]
    scored = []
    for r in cc_records:
        ppl = get_perplexity(r['text'])
        if ppl:
            scored.append((ppl, r))

    scored.sort(key=lambda x: x[0])

    print("=" * 70)
    print("LOW PERPLEXITY (high quality) — top 3 Common Crawl documents")
    print("=" * 70)
    for ppl, r in scored[:3]:
        print(f"\nPerplexity: {ppl:.1f}")
        print(f"URL: {r.get('url', 'N/A')}")
        print(r['text'][:300])
        print("...")

    print()
    print("=" * 70)
    print("HIGH PERPLEXITY (low quality) — bottom 3 Common Crawl documents")
    print("=" * 70)
    for ppl, r in scored[-3:]:
        print(f"\nPerplexity: {ppl:.1f}")
        print(f"URL: {r.get('url', 'N/A')}")
        print(r['text'][:300])
        print("...")
else:
    print("KenLM model not found — skipping perplexity sampling")

## 7. Terminal punctuation analysis

In [ ]:
def terminal_punct_ratio(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if not lines:
        return 0.0
    terminal = sum(1 for l in lines if l.endswith(('.', '!', '?', '"', "'")))
    return terminal / len(lines)

ratios_by_source = {s: [] for s in ['wikipedia', 'code', 'common_crawl']}
for r in curated:
    src = r.get('source', 'unknown')
    if src in ratios_by_source:
        ratios_by_source[src].append(terminal_punct_ratio(r['text']))

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#4C72B0', '#55A868', '#DD8452']
for (source, ratios), color in zip(ratios_by_source.items(), colors):
    if ratios:
        ax.hist(ratios, bins=40, alpha=0.6, color=color, label=source, edgecolor='white')

ax.set_xlabel('Terminal punctuation ratio')
ax.set_ylabel('Count')
ax.set_title('Terminal Punctuation Ratio by Source', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("Mean terminal punctuation ratio:")
for source, ratios in ratios_by_source.items():
    if ratios:
        print(f"  {source:<20} {np.mean(ratios):.3f}")

## 8. Dataset size reduction summary

In [ ]:
if stats_path.exists() and validated:
    curated_tokens = sum(len(r['text']) for r in curated) // 4
    validated_tokens = sum(len(r['text']) for r in validated) // 4

    stages = ['Raw curated', 'After validation']
    doc_counts = [stats['total'], stats['kept']]
    token_ests = [
        stats['total'] * 750,   # rough est: avg 750 tokens per doc
        stats['kept'] * 750,
    ]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].bar(stages, doc_counts, color=['#DD8452', '#55A868'], alpha=0.85, edgecolor='white')
    axes[0].set_title('Document Count', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Documents')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    for i, v in enumerate(doc_counts):
        axes[0].text(i, v + max(doc_counts)*0.01, f'{v/1e6:.2f}M', ha='center', fontsize=10)

    axes[1].bar(stages, token_ests, color=['#DD8452', '#55A868'], alpha=0.85, edgecolor='white')
    axes[1].set_title('Estimated Tokens', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Tokens')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
    for i, v in enumerate(token_ests):
        axes[1].text(i, v + max(token_ests)*0.01, f'{v/1e9:.2f}B', ha='center', fontsize=10)

    fig.suptitle('Dataset Size — Before vs After Validation', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    retention = 100 * stats['kept'] / max(stats['total'], 1)
    print(f"Retention rate: {retention:.1f}%")
    print(f"Removed: {stats['total'] - stats['kept']:,} documents ({100-retention:.1f}%)")